# ⚡ Amazon ML Challenge 2026 — Kaggle CPU Pipeline

This notebook is specifically designed to maximize safety and stream live output on Kaggle Notebooks without crashing.

## 🛠️ Step 1: System Hardware Check

In [ ]:
import psutil, os
import torch
ram_gb = psutil.virtual_memory().total / (1024 ** 3)
cpus = os.cpu_count()
gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
print(f"KAGGLE HARDWARE: {cpus} CPU Cores | {ram_gb:.2f} GB RAM | {gpus} GPUs")

## 📦 Step 2: Clone Repo, Pull LFS, & Free Up 7.5GB Disk Space

In [ ]:
import os

# 1. Clean up Kaggle working directory in case of previous failed runs
!rm -rf /kaggle/working/AmazonMLChallenge2026

# 2. Install Git LFS and Clone
!apt-get install -y git-lfs > /dev/null
!git lfs install
!git clone https://github.com/priyanshu-ogdev/AmazonMLChallenge2026.git
%cd AmazonMLChallenge2026

# 3. Pull LFS files (temporarily spikes disk usage to ~11 GB)
!git lfs pull

# 4. IMMEDIATELY delete the .git folder (frees up ~5.3 GB of redundant LFS cache!)
!rm -rf .git

# 5. Recombine the split 2.4GB index and delete the parts (frees up another 2.4 GB!)
!cat output/phase1_blocking_train/candidate_index.pkl.part* > output/phase1_blocking_train/candidate_index.pkl
!rm output/phase1_blocking_train/candidate_index.pkl.part*

print("\n✅ Git LFS Pull and Workspace Optimization Complete!")
!df -h /kaggle/working


## ⚙️ Step 3: Install Dependencies

In [ ]:
!pip install -q -r code/business_entity_resolution/requirements.txt
!chmod +x scripts/*.sh
import os
os.environ['DISABLE_LOGGING'] = 'true'
!bash scripts/00_verify_environment.sh --python-path python3

## 🚀 Step 4: Full CPU Pipeline Execution
Runs Phase 1 (Blocking), Phase 2 (Lexical Features), Phase 3 (Scoring), and Phase 4 (Inference).
By using `DISABLE_LOGGING=true` and direct shell bangs, the output streams flawlessly in Kaggle.

In [ ]:
import os
os.environ['DISABLE_LOGGING'] = 'true'
os.environ['PYTHON_PATH'] = 'python3'

print("==================================================")
print(" STARTING ENTIRE CPU PIPELINE")
print("==================================================")
!bash scripts/run_all_phases.sh --run-mode Full --skip-gpu --python-path python3

## 📦 Step 5: Final Kaggle Output Generation

In [ ]:
import os

zip_name = "/kaggle/working/final_submission_artifacts.zip"
!zip -r -q {zip_name} output/ -x "output/**/_tmp_worker_chunks/*" "output/**/*.tmp"
print(f"Final package saved to {zip_name}. Download it from the right-hand Kaggle output panel!")